⚡ Intermédiaire | ⏱ 90 min | 🔑 `__eq__`, `__lt__`, `__add__`, `__len__`, `__getitem__`, `__hash__`, try/except, raise, exception personnalisée

# 07 - Méthodes spéciales et exceptions

À la fin : tes objets se comparent, s'additionnent, se trient et s'itèrent comme les types natifs, et tes méthodes signalent proprement les erreurs avec des exceptions.


## 1. Méthodes spéciales : brancher tes objets sur la syntaxe Python

Les méthodes `__nom__` (*dunder*, pour *double underscore*) sont appelées par Python derrière un opérateur ou une fonction native. Tu en connais déjà deux : `__init__` et `__repr__`.

| Syntaxe | Méthode appelée |
|---|---|
| `a == b`, `a != b` | `__eq__`, `__ne__` |
| `a < b`, `a <= b`, `a > b`, `a >= b` | `__lt__`, `__le__`, `__gt__`, `__ge__` |
| `a + b`, `a - b`, `a * n` | `__add__`, `__sub__`, `__mul__` |
| `len(a)` | `__len__` |
| `a[i]`, `for x in a` | `__getitem__`, `__iter__` |
| `x in a` | `__contains__` |
| `if a:`, `bool(a)` | `__bool__` |
| `hash(a)`, clé de dict, élément de set | `__hash__` |
| `a(...)` | `__call__` |


## 2. Égalité et hachage


In [ ]:
# Par défaut, == compare l'identité : deux objets au même état sont « différents »
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

print(Point(1, 2) == Point(1, 2))


In [ ]:
# __eq__ : définir l'égalité par l'état
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, autre):
        if not isinstance(autre, Point):
            return NotImplemented           # laisse Python essayer l'autre opérande
        return (self.x, self.y) == (autre.x, autre.y)

print(Point(1, 2) == Point(1, 2), Point(1, 2) == (1, 2))


In [ ]:
# Définir __eq__ rend l'objet non hachable : plus de set ni de clé de dict
try:
    {Point(1, 2)}
except TypeError as e:
    print("TypeError :", e)


In [ ]:
# __hash__ cohérent avec __eq__ : même état, même hash
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, autre):
        if not isinstance(autre, Point):
            return NotImplemented
        return (self.x, self.y) == (autre.x, autre.y)

    def __hash__(self):
        return hash((self.x, self.y))

print({Point(1, 2), Point(1, 2), Point(0, 0)})     # les doublons disparaissent


## 3. Ordre : trier des objets

`__lt__` (*less than*) suffit à `sorted()`, `min()` et `max()`. Pour avoir tous les opérateurs, `functools.total_ordering` déduit les autres à partir de `__eq__` et `__lt__`.


In [ ]:
# __lt__ suffit pour sorted(), min(), max()
class Etudiant:
    def __init__(self, nom, moyenne):
        self.nom, self.moyenne = nom, moyenne

    def __repr__(self):
        return f"{self.nom}({self.moyenne})"

    def __lt__(self, autre):
        return self.moyenne < autre.moyenne

promo = [Etudiant("Ana", 14), Etudiant("Bo", 17), Etudiant("Cy", 11)]
print(sorted(promo), max(promo))


In [ ]:
# functools.total_ordering complète <=, >, >= à partir de __eq__ et __lt__
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, texte):
        self.parts = tuple(int(p) for p in texte.split("."))

    def __eq__(self, autre):
        return self.parts == autre.parts

    def __lt__(self, autre):
        return self.parts < autre.parts

print(Version("3.10") > Version("3.9"), Version("1.0") >= Version("1.0"))


## 4. Opérateurs arithmétiques

`__add__` et compagnie doivent **renvoyer un nouvel objet**, pas modifier `self`. `__rmul__` gère le cas où l'objet est à droite (`3 * v`).


In [ ]:
# __add__ et __mul__ : Vecteur + Vecteur, Vecteur * nombre
class Vecteur:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vecteur({self.x}, {self.y})"

    def __add__(self, autre):
        return Vecteur(self.x + autre.x, self.y + autre.y)

    def __mul__(self, k):
        return Vecteur(self.x * k, self.y * k)

    __rmul__ = __mul__                      # permet aussi 3 * v

v = Vecteur(1, 2)
print(v + Vecteur(10, 10), v * 3, 3 * v)


## 5. Se comporter comme une collection

Avec `__len__`, `__getitem__` et `__contains__`, un objet se manipule comme une liste : `len(p)`, `p[0]`, `for x in p`, `x in p`. `__getitem__` seul rend déjà l'objet itérable.


In [ ]:
# __len__, __getitem__, __contains__ : la playlist se manipule comme une liste
class Playlist:
    def __init__(self, titres):
        self._titres = list(titres)

    def __len__(self):
        return len(self._titres)

    def __getitem__(self, i):
        return self._titres[i]              # rend aussi l'objet itérable et découpable

    def __contains__(self, titre):
        return titre in self._titres

p = Playlist(["intro", "refrain", "final"])
print(len(p), p[0], p[-1], "refrain" in p)
for titre in p:
    print("-", titre)


In [ ]:
# __bool__ : if panier: ; __call__ : l'objet s'utilise comme une fonction
class Panier:
    def __init__(self):
        self.articles = []

    def __bool__(self):
        return len(self.articles) > 0

class Multiplicateur:
    def __init__(self, k):
        self.k = k

    def __call__(self, x):
        return x * self.k

triple = Multiplicateur(3)
print(bool(Panier()), triple(14))


## 6. Exceptions : signaler une erreur

Une **exception** interrompt l'exécution et remonte jusqu'à un `try` qui la capture. Dans une classe, `raise ValueError("...")` est la bonne façon de refuser un argument invalide : l'appelant ne peut pas l'ignorer, contrairement à un `return False` ou un `print`.

Exceptions courantes, toutes filles d'`Exception` : `ValueError` (bonne forme, mauvaise valeur), `TypeError` (mauvais type), `KeyError`, `IndexError`, `AttributeError`, `ZeroDivisionError`, `FileNotFoundError`.

Structure complète : `try` (code surveillé) / `except` (si erreur) / `else` (si pas d'erreur) / `finally` (dans tous les cas).


In [ ]:
# raise : refuser un argument invalide ; l'appelant choisit comment réagir
class Compte:
    def __init__(self, solde=0):
        self.solde = solde

    def retirer(self, montant):
        if montant <= 0:
            raise ValueError(f"montant invalide : {montant}")
        if montant > self.solde:
            raise ValueError("solde insuffisant")
        self.solde -= montant

c = Compte(100)
try:
    c.retirer(500)
except ValueError as e:
    print("Refusé :", e)
print(c.solde)


In [ ]:
# except ciblé, else si tout s'est bien passé, finally dans tous les cas
def convertir(texte):
    try:
        valeur = int(texte)
    except ValueError:
        print(f"{texte!r} n'est pas un entier")
        return None
    else:
        print("conversion OK")
        return valeur
    finally:
        print("fin de convertir")

print(convertir("42"))
print(convertir("abc"))


In [ ]:
# Plusieurs except : du plus précis au plus général
def lire(d, cle):
    try:
        return 100 / d[cle]
    except KeyError:
        return "clé absente"
    except ZeroDivisionError:
        return "division par zéro"
    except Exception as e:                   # filet de sécurité, toujours en dernier
        return f"autre erreur : {type(e).__name__}"

print(lire({"a": 4}, "a"), lire({}, "a"), lire({"a": 0}, "a"), lire({"a": "x"}, "a"))


## 7. Exceptions personnalisées

Une exception est une classe qui hérite d'`Exception`. Créer les siennes permet à l'appelant de capturer **précisément** les erreurs métier (`SoldeInsuffisant`) sans les confondre avec un `ValueError` venu d'ailleurs. On peut y ranger des données utiles.


In [ ]:
# Hiérarchie d'exceptions métier : capturer précisément ou globalement
class ErreurBancaire(Exception):
    """Base de toutes les erreurs du module bancaire."""

class SoldeInsuffisant(ErreurBancaire):
    def __init__(self, manque):
        super().__init__(f"il manque {manque} €")
        self.manque = manque

class Compte:
    def __init__(self, solde=0):
        self.solde = solde

    def retirer(self, montant):
        if montant > self.solde:
            raise SoldeInsuffisant(montant - self.solde)
        self.solde -= montant

try:
    Compte(100).retirer(130)
except SoldeInsuffisant as e:
    print(e, "| manque =", e.manque)
except ErreurBancaire:
    print("autre erreur bancaire")


In [ ]:
# raise ... from : convertir une erreur technique en erreur métier sans perdre la cause
class ConfigInvalide(Exception):
    pass

def charger_port(config):
    try:
        return int(config["port"])
    except (KeyError, ValueError) as e:
        raise ConfigInvalide("port manquant ou non numérique") from e

try:
    charger_port({"port": "abc"})
except ConfigInvalide as e:
    print(e, "| cause :", repr(e.__cause__))


Style Python : **EAFP** (*easier to ask forgiveness than permission*) : on tente, on capture. Plutôt que **LBYL** (*look before you leap*) : tout vérifier avant.


In [ ]:
# LBYL vérifie avant ; EAFP tente et capture (plus court, un seul accès)
d = {"a": 1}

valeur = d["b"] if "b" in d else 0            # LBYL

try:                                          # EAFP
    valeur = d["b"]
except KeyError:
    valeur = 0
print(valeur)


## ⚠️ Pièges

- `__eq__` sans `__hash__` : l'objet devient non hachable (plus de set ni de clé de dict).
- `__repr__` sans `!r` : `Livre(Dune, Herbert)` au lieu de `Livre('Dune', 'Herbert')`.
- `__add__` qui modifie `self` au lieu de renvoyer un nouvel objet.
- `except:` nu ou `except Exception: pass` : avale tout, y compris `Ctrl+C` et tes propres bugs. Toujours cibler, toujours faire quelque chose (message, `raise`).
- Renvoyer un code d'erreur (`-1`, `None`, `False`) là où une exception s'impose : l'appelant l'oublie.
- Lever `Exception("...")` tout court : l'appelant ne peut rien cibler.


In [ ]:
# BUG classique : l'erreur est avalée, le programme continue avec une valeur fausse
def moyenne(notes):
    try:
        return sum(notes) / len(notes)
    except:                                  # ❌ attrape tout, cache le problème
        pass

print(moyenne([]))                           # None silencieux au lieu d'une erreur claire


In [ ]:
# Correction : refuser explicitement le cas invalide
def moyenne(notes):
    if not notes:
        raise ValueError("liste de notes vide")
    return sum(notes) / len(notes)

try:
    moyenne([])
except ValueError as e:
    print("ValueError :", e)
